# Deleting Data in Mongoose

## Common delete methods

| Method | Returns | Runs middleware |
|---|---|---|
| `deleteOne(filter)` | Result metadata | `pre/post('deleteOne')` query hooks |
| `deleteMany(filter)` | Result metadata | `pre/post('deleteMany')` query hooks |
| `findOneAndDelete(filter)` | The deleted document, or `null` | `pre/post('findOneAndDelete')` |
| `findByIdAndDelete(id)` | The deleted document, or `null` | Same, matched by `_id` |
| `doc.deleteOne()` | The document | Document-level `deleteOne` hooks, with `this` as the document |

Metadata shape from `deleteOne` / `deleteMany`:

```javascript
{ acknowledged: true, deletedCount: 1 }
```

`deletedCount: 0` is not an error — the filter simply matched nothing. It's how you distinguish a real deletion from a request for something that was never there, which is exactly what a 404 vs 204 decision hangs on.

## Examples

### Delete a single document by condition

```javascript
// Deletes the first user with the name "John"
await User.deleteOne({ name: 'John' });
```

### Delete multiple documents

```javascript
// Deletes all users who have an inactive status
await User.deleteMany({ status: 'inactive' });
```

### Find by ID and delete

```javascript
const deletedUser = await User.findByIdAndDelete(userId);
if (!deletedUser) return res.status(404).send('Not found');
```

Use this form whenever you need the removed data — to log it, to return it, or to clean up related documents that reference it.

### In an Express route

```javascript
app.delete('/movies/:id', async (req, res) => {
  const movie = await Movie.findByIdAndDelete(req.params.id);
  if (!movie) return res.status(404).json({ error: 'Movie not found' });
  res.status(204).send();   // 204 No Content: success, nothing to return
});
```

`204` is the conventional response for a successful delete with no body. Return `200` with the deleted object instead if the client actually needs it.

## Deprecated methods

`remove()`, `findByIdAndRemove()` and `findOneAndRemove()` were removed in Mongoose 7. If a tutorial uses them, it predates v7 — the replacements are `deleteOne()`/`deleteMany()` and `findByIdAndDelete()`/`findOneAndDelete()`.

## Middleware and cascading

Deleting a document doesn't touch anything referencing it. MongoDB has no foreign keys and no `ON DELETE CASCADE`, so orphaned references are yours to manage.

```javascript
// Clean up a user's posts when the user is deleted
userSchema.post('findOneAndDelete', async function (doc) {
  if (doc) await Post.deleteMany({ author: doc._id });
});
```

The `if (doc)` guard matters — the hook fires even when nothing matched, and `doc` is `null` in that case.

Two traps here:

- **Query vs document hooks.** `schema.pre('deleteOne')` defaults to the *query* variant, where `this` is the query and the document isn't loaded. For the document form, register it explicitly: `schema.pre('deleteOne', { document: true, query: false }, fn)`.
- **`deleteMany` gives you no documents.** A post-hook on `deleteMany` receives only the result metadata, so cascade logic has to query for the affected ids *before* deleting.

## Soft deletes

Often the better default. Instead of removing the row, flag it:

```javascript
// schema
deletedAt: { type: Date, default: null }

// "delete"
await Movie.findByIdAndUpdate(id, { deletedAt: new Date() });

// every read must now exclude them
await Movie.find({ deletedAt: null });
```

You keep an audit trail and can undo mistakes, at the cost of remembering the filter in every query. A `pre('find')` hook that injects `{ deletedAt: null }` automatically saves you from forgetting it. Alternatively, set a TTL index so soft-deleted records purge themselves after a retention window:

```javascript
schema.index({ deletedAt: 1 }, { expireAfterSeconds: 60 * 60 * 24 * 30 });
```

## Gotchas

- **`deleteMany({})` empties the collection.** No confirmation, no undo. The empty filter is valid and matches everything — the single most destructive typo available in Mongoose. Some people write `deleteMany({ _id: { $exists: true } })` in scripts purely so the intent is explicit.
- **Never spread `req.body` or `req.query` into a delete filter.** `{ _id: { $ne: null } }` arriving as a query param deletes the whole collection.
- **Invalid ObjectId throws a `CastError`** on `findByIdAndDelete`, rather than returning `null`. Guard with `isValid()` or handle it in your error middleware, or a malformed URL becomes a 500 instead of a 404.
- **Deleting a collection vs its documents.** `deleteMany({})` leaves the (now empty) collection and its indexes in place; `Model.collection.drop()` removes the collection entirely. `show collections` in mongosh will still list an emptied collection.
- **Deletes aren't transactional across documents** unless wrapped in a session. A cascade that fails halfway leaves partial state.
- **Reclaiming disk space.** MongoDB doesn't return freed space to the OS on delete; the files stay sized until a compact or resync.

## Sources

- [Model API — deleteOne / deleteMany](https://mongoosejs.com/docs/api/model.html)
- [Mongoose middleware](https://mongoosejs.com/docs/middleware.html)
- [MongoDB delete documents](https://www.mongodb.com/docs/manual/tutorial/remove-documents/)